In [ ]:
import yasa
edf_path = yasa.fetch_sample("night_young.edf")
hypno_path = yasa.fetch_sample("night_young_hypno.csv")

In [ ]:
import mne
raw = mne.io.read_raw_edf(edf_path, preload=True)
raw

In [ ]:
print(raw.ch_names)

In [ ]:
raw.drop_channels(["ROC-A1", "LOC-A2", "EMG1-EMG2", "EKG-R-EKG-L"])
chan = raw.ch_names
print(chan)

In [ ]:
raw.resample(100)
sf = raw.info["sfreq"]
sf

In [ ]:
raw.filter(0.3, 45)

In [ ]:
data = raw.get_data(units="uV")
print(data.shape)

In [ ]:
import pandas as pd
import yasa
hypno = pd.read_csv(hypno_path).squeeze().to_numpy()
hyp = yasa.Hypnogram.from_integers(hypno, freq="30s", scorer="Expert")
hyp

In [ ]:
hyp.plot_hypnogram();

In [ ]:
hyp.sleep_statistics()

In [ ]:
counts, probs = hyp.transition_matrix()
probs.round(3)

In [ ]:
import numpy as np
np.diag(probs.loc["N2":, "N2":]).mean().round(3)

In [ ]:
hypno_up = hyp.upsample_to_data(data=raw)
print(len(hypno_up))

In [ ]:
yasa.plot_spectrogram(data[chan.index("C4-A1")], sf, hypno_up);

In [ ]:
yasa.bandpower(raw)

In [ ]:
yasa.bandpower(raw, relative=False, bands=[(1, 9, "Slow"), (9, 30, "Fast")])

In [ ]:
bandpower = yasa.bandpower(raw, hypno=hypno_up, include=(2, 3, 4))

In [ ]:
fig = yasa.topoplot(bandpower.xs(3)["Delta"])

In [ ]:
sp = yasa.spindles_detect(raw, hypno=hypno_up, include=(2, 3))

In [ ]:
sp.summary()

In [ ]:
sp.summary(grp_chan=True, grp_stage=True)

In [ ]:
# Because of the large number of channels, we disable the 95%CI and legend
sp.plot_average(ci=None, legend=False, palette="Blues");

In [ ]:
sw = yasa.sw_detect(raw, hypno=hypno_up, include=(2, 3))
sw.summary()

In [ ]:
sw.plot_average(ci=None, legend=False, palette="Blues");

In [ ]:
sls = yasa.SleepStaging(raw, eeg_name="C3-A2")
hypno_pred = sls.predict()  # Returns a yasa.Hypnogram
yasa.plot_hypnogram(hypno_pred);  # Plot

In [ ]:
ebe = hyp.evaluate(hypno_pred)
ebe.get_agreement()